# manhattan-heuristic

**What does a badly inadmissible heuristic buy, and what does it cost?**

A\* scores each candidate `f = g + h`, and returns an optimal path only if `h`
never overestimates the cost still to go. `experiments/heuristic-admissibility`
established that this project's great-circle heuristic clears that bar, and
that Vincenty — the *more accurate* formula — would not. Vincenty misses by
tens of kilometres on 58.6% of edges: a precision-scale failure, and the only
kind the project had ever measured.

This experiment measures the other kind. Manhattan distance is the textbook
grid heuristic: walk one axis, then the other. It is admissible and excellent
on a grid of unit moves. An airline network has coordinates but no grid — the
vertices are airports and the moves between them are arbitrary flight legs
weighted in great-circle kilometres — so a taxicab estimate walks a meridian
and a parallel to reach a goal the search will actually reach along an arc.
It comes in high.

The question is not *whether* that breaks admissibility; two legs of a right
triangle are never shorter than the hypotenuse, so it must. The question is
what happens next, and it has two halves that pull against each other:

1. **What it costs.** An overestimate lets A\* close a vertex the optimal path
   ran through, and `flight_planner.pathfinding.astar` never reopens one. So it
   returns a longer route and reports it as optimal. How often, and by how
   much?
2. **What it buys.** That same overestimate makes A\* greedier, and a greedier
   search expands fewer vertices. How many fewer?

The answer to the second is why this is worth measuring rather than only
warning about, and the two together are a bound: if the estimate never exceeds
the truth by more than a factor of `r`, the returned cost cannot exceed `r ×`
optimal. Section 2 measures `r`. Section 4 measures how much of it the data
actually spends.

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a different
answer.

## Setup

Only the first cell differs between Colab and a local checkout.

In [ ]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/<owner>/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima matplotlib
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit("install the package first -- see the comment above") from error

In [ ]:
import random
import statistics
from pathlib import Path

import matplotlib.pyplot as plt

from flight_planner import AStar, Dijkstra
from flight_planner.experiments import Experiment
from flight_planner.geo import (
    Haversine,
    Manhattan,
    Point,
    haversine_heuristic,
    manhattan_heuristic,
)

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters
epsilon = parameters["epsilon_km"]
parameters

## The data

Opening the snapshot re-hashes every file against the manifest. Nothing is
narrowed: the question is about the whole world network, and a slice would
make the suboptimality rate a statement about that slice instead.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria)

catalog = experiment.catalog()
routes, airports = catalog.routes, catalog.airports
print(f"{len(airports):,} airports  {len(routes):,} routes")

## 1. The formula, and why it looks reasonable

On a grid, Manhattan distance is the sum of the axis-aligned legs. The
lat/lon analogue is one leg along a meridian and one along a parallel:

$$h = R \left( |\Delta\varphi| + |\Delta\lambda| \cos\frac{\varphi_1 + \varphi_2}{2} \right)$$

Two details that are easy to get wrong, both handled in
`flight_planner.geo.manhattan`:

* The latitude leg is exact — a meridian is a great circle. The longitude leg
  is not: a parallel is a *small* circle whose radius shrinks with latitude, so
  it is scaled by the cosine of the mean latitude. That is an approximation
  twice over, since the two points sit at different latitudes.
* The longitude difference is folded to the short way round. Without that an
  antimeridian pair is estimated going the long way, and the overestimate stops
  being a property of the L1 norm and becomes a bug.

`Manhattan` uses the same 6,371.0 km sphere as `Haversine`, deliberately: the
two differ in *shape* and not in radius, so nothing below is confounded by a
disagreement about the size of the Earth.

Note what this formula is. It is `Equirectangular` from
`src/demos/custom_distance_formula_example.py` with the L1 norm in place of L2,
and since $|x| + |y| \ge \sqrt{x^2 + y^2}$, Manhattan is never below that
flat-plane approximation. The demo presents its formula as a precision trade.
This section is the beginning of the argument that the L1 version is not a
precision trade at all.

In [ ]:
# One pair, by hand, through Point.distance_to -- the same Strategy seam every
# formula plugs into.
jfk = Point(40.6413, -73.7781)
ord_ = Point(41.9742, -87.9073)

for name, formula in (("Haversine", Haversine()), ("Manhattan", Manhattan())):
    print(f"{name:>10}  JFK -> ORD  {jfk.distance_to(ord_, formula):8.2f} km")

# And a pair across the antimeridian, where the wrap earns its comment.
syd = Point(-33.9461, 151.1772)
print()
for name, formula in (("Haversine", Haversine()), ("Manhattan", Manhattan())):
    print(f"{name:>10}  SYD -> JFK  {syd.distance_to(jfk, formula):8.2f} km")

## 2. Per-edge admissibility, over every edge

The single-edge case is the one that can actually fail, and it is the one
`heuristic-admissibility` reports for the real heuristic — so this reports the
same four numbers, the same way, and the two are directly comparable.

An *excess* is `estimate - weight`: positive means the estimate is above a
value it must not exceed. The full edge set costs a fraction of a second, so
nothing here is sampled.

The `estimate / weight` ratio is the addition. It is the quantity that bounds
what section 4 can find: A\* guided by a heuristic that never exceeds the truth
by more than a factor of `r` cannot return a path costing more than `r ×`
optimal.

In [ ]:
def summarise(excesses, epsilon, total):
    """Reduce signed `estimate - truth` excesses to the four numbers that matter.

    The same reduction `heuristic-admissibility` applies, so the two
    experiments' outputs can be read side by side.

    Args:
        excesses: Signed `estimate - truth` per edge, in kilometres.
        epsilon: Below this, a positive excess is float noise, not a breach.
        total: Population the excesses were drawn from.

    Returns:
        Mapping of the violation count, its share, the worst excess, and the
        largest margin by which an estimate came in under the truth.
    """
    violations = [value for value in excesses if value > epsilon]
    return {
        "checked": total,
        "violations": len(violations),
        "violation_rate": len(violations) / total if total else 0.0,
        "worst_excess_km": max(excesses) if excesses else 0.0,
        "largest_understatement_km": -min(excesses) if excesses else 0.0,
    }


def quantiles(values):
    """Return min, median, p99 and max of `values`.

    Args:
        values: Numbers to describe. Not modified.

    Returns:
        Mapping of the four positions.
    """
    ordered = sorted(values)
    def at(q):
        return ordered[min(len(ordered) - 1, int(q * len(ordered)))]
    return {
        "min": ordered[0],
        "median": at(0.5),
        "p99": at(0.99),
        "max": ordered[-1],
    }

In [ ]:
manhattan = Manhattan()

excesses = []
ratios = []
for route in routes:
    estimate = manhattan.calculate(route.origin, route.destination)
    excesses.append(estimate - route.distance_km)
    # A ratio needs a denominator. The snapshot contains one zero-weight edge
    # -- see below -- so it is rated by the excess above and not here.
    if route.distance_km > 0.0:
        ratios.append(estimate / route.distance_km)

per_edge = summarise(excesses, epsilon, len(routes))
worst_edge = routes[excesses.index(max(excesses))]
per_edge["worst_edge"] = {
    "origin": worst_edge.origin.iata_code,
    "destination": worst_edge.destination.iata_code,
    "weight_km": worst_edge.distance_km,
    "estimate_km": manhattan.calculate(worst_edge.origin, worst_edge.destination),
}
per_edge["ratio"] = quantiles(ratios)
per_edge["ratio"]["edges_rated"] = len(ratios)
per_edge["zero_weight_edges"] = len(routes) - len(ratios)

print(
    f"{per_edge['violations']:,} of {per_edge['checked']:,} edges "
    f"({per_edge['violation_rate']:.3%}) estimate above their own weight"
)
print(f"worst excess {per_edge['worst_excess_km']:,.2f} km at {per_edge['worst_edge']}")
print(f"ratio estimate/weight {per_edge['ratio']}")

Every edge but one, and the exception is not a near miss. The snapshot contains
a single zero-weight edge — a self-loop, an airport with a scheduled flight to
itself — where the estimate is zero too. So the formula clears the bar on
exactly the one edge where every formula clears it, which is worth saying out
loud: the 99.998% is a real rate and not a rounding artifact.

Compare what the same reduction reports for the admissible heuristic: **0
violations of 66,332 edges**, with the largest excess of any estimate over its
own weight at 1.8e-12 km — the last bit of a float, because there the estimate
and the weight are the same computation run twice. This is not that. The median
edge is overestimated by a third.

In [ ]:
# The one edge that does not violate, named rather than left as an arithmetic
# residue.
for route in routes:
    if route.distance_km == 0.0:
        print(
            f"{route.flight_number}  {route.origin.iata_code} -> "
            f"{route.destination.iata_code}  weight {route.distance_km} km  "
            f"estimate {manhattan.calculate(route.origin, route.destination)} km"
        )

## 3. Consistency

Admissibility is the textbook condition, but it is not the one this
implementation needs. `flight_planner.pathfinding.astar` closes a vertex when
it is popped and never reopens it, so what it requires is *consistency* —
`h(u) <= w(u, v) + h(v)` for every edge — which is the stronger property.
`astar-consistency` is the experiment that establishes the difference.

For a heuristic that fails section 2 this cannot possibly hold, so the count is
not the point. The *magnitude* is: the worst breach of the triangle inequality
is the largest single-step error the search is acting on when it decides which
vertex to close.

Every edge, against a seeded sample of goals. All 3,387 goals would be 224.6
million checks for a property that does not vary by goal; a hundred is 6.6
million, in about nine seconds.

In [ ]:
taxicab_h = manhattan_heuristic(memoize=False)
sampled_goals = random.Random(parameters["goal_sample_seed"]).sample(
    list(airports), min(parameters["consistency_goals"], len(airports))
)

checks = 0
violations = 0
worst_violation = 0.0
for goal in sampled_goals:
    for route in routes:
        checks += 1
        # Positive slack means the inequality holds with room to spare.
        slack = (
            route.distance_km
            + taxicab_h(route.destination, goal)
            - taxicab_h(route.origin, goal)
        )
        if slack < -epsilon:
            violations += 1
        worst_violation = max(worst_violation, -slack)

consistency = {
    "goals_sampled": len(sampled_goals),
    "goals_available": len(airports),
    "checks": checks,
    "violations": violations,
    "violation_rate": violations / checks,
    "worst_violation_km": worst_violation,
}
print(
    f"{violations:,} of {checks:,} checks ({violations / checks:.1%}) breach "
    f"the triangle inequality, worst by {worst_violation:,.1f} km"
)

## 4. What it buys, and what it costs

Three searches per pair, over the same graph: `Dijkstra` for the optimum,
`AStar(haversine_heuristic())` for the admissible baseline, and
`AStar(manhattan_heuristic())` for the formula under test. All three go through
`FlightPlanner.search_route`, so nothing here reaches past the public seam.

The pairs are drawn uniformly at random from the snapshot's airports, from a
recorded seed. The headline number is a *rate* — how often the answer is wrong
— and a hand-picked list cannot support one. Section 4b shows concretely why
that matters.

A\*'s cost against Dijkstra's is also the control: the admissible run must
match Dijkstra to the last bit, and it is asserted rather than hoped for. If
that assertion ever fires, the harness is broken and nothing below means
anything.

In [ ]:
planner = catalog.planner()

ALGORITHMS = ("dijkstra", "haversine", "manhattan")


def compare(pairs):
    """Run all three searches over each pair and tabulate what they did.

    Args:
        pairs: Iterable of `(origin_iata, destination_iata)`.

    Returns:
        Tuple of (rows, unreachable). One row per pair with a route; pairs with
        no route at all are returned separately rather than dropped, because a
        silent skip reads as full coverage.
    """
    rows = []
    unreachable = []
    for origin, destination in pairs:
        optimal = planner.search_route(origin, destination, algorithm=Dijkstra())
        if not optimal.found:
            unreachable.append([origin, destination])
            continue

        admissible = planner.search_route(
            origin, destination, algorithm=AStar(haversine_heuristic())
        )
        taxicab = planner.search_route(
            origin, destination, algorithm=AStar(manhattan_heuristic())
        )

        # The control. An admissible, consistent heuristic must return
        # Dijkstra's answer exactly; if it does not, the measurement is wrong
        # rather than the finding being interesting.
        assert abs(admissible.cost - optimal.cost) <= epsilon, (origin, destination)

        results = {"dijkstra": optimal, "haversine": admissible, "manhattan": taxicab}
        rows.append(
            {
                "origin": origin,
                "destination": destination,
                "optimal_km": optimal.cost,
                "manhattan_km": taxicab.cost,
                "excess": (taxicab.cost - optimal.cost) / optimal.cost,
                "legs_optimal": len(optimal.path),
                "legs_manhattan": len(taxicab.path),
                "expanded": {k: results[k].nodes_expanded for k in ALGORITHMS},
                "pushed": {k: results[k].nodes_pushed for k in ALGORITHMS},
                "peak_frontier": {k: results[k].peak_frontier for k in ALGORITHMS},
            }
        )
    return rows, unreachable

In [ ]:
codes = sorted(catalog.iata_codes)
rng = random.Random(parameters["pair_seed"])
sampled_pairs = [tuple(rng.sample(codes, 2)) for _ in range(parameters["pair_sample"])]

rows, unreachable = compare(sampled_pairs)
print(f"{len(rows)} pairs with a route, {len(unreachable)} unreachable and skipped")

In [ ]:
excess = [row["excess"] for row in rows]
suboptimal = [row for row in rows if row["excess"] > epsilon]
worst_row = max(rows, key=lambda row: row["excess"])

sample = {
    "pairs_requested": parameters["pair_sample"],
    "pairs_with_a_route": len(rows),
    "pairs_unreachable": len(unreachable),
    "unreachable_pairs": unreachable,
    "suboptimal": len(suboptimal),
    "suboptimal_rate": len(suboptimal) / len(rows),
    "excess": {
        "mean": statistics.mean(excess),
        "median": statistics.median(excess),
        "max": max(excess),
    },
    "worst_pair": worst_row,
    "mean_expanded": {
        key: statistics.mean(row["expanded"][key] for row in rows) for key in ALGORITHMS
    },
    "mean_pushed": {
        key: statistics.mean(row["pushed"][key] for row in rows) for key in ALGORITHMS
    },
    "mean_peak_frontier": {
        key: statistics.mean(row["peak_frontier"][key] for row in rows)
        for key in ALGORITHMS
    },
    # Per-pair ratios rather than a ratio of the means, so one enormous query
    # cannot carry the number.
    "mean_expansion_ratio": {
        "manhattan_over_haversine": statistics.mean(
            row["expanded"]["manhattan"] / row["expanded"]["haversine"] for row in rows
        ),
        "manhattan_over_dijkstra": statistics.mean(
            row["expanded"]["manhattan"] / row["expanded"]["dijkstra"] for row in rows
        ),
    },
}

print(
    f"suboptimal on {sample['suboptimal']} of {sample['pairs_with_a_route']} "
    f"pairs ({sample['suboptimal_rate']:.1%})"
)
print(
    f"cost excess  mean {sample['excess']['mean']:.2%}  "
    f"median {sample['excess']['median']:.2%}  max {sample['excess']['max']:.2%}"
)
print(f"mean nodes expanded  {sample['mean_expanded']}")
print(f"mean expansion ratio {sample['mean_expansion_ratio']}")
print(
    f"worst pair  {worst_row['origin']} -> {worst_row['destination']}  "
    f"{worst_row['optimal_km']:,.1f} km in {worst_row['legs_optimal']} legs "
    f"becomes {worst_row['manhattan_km']:,.1f} km in {worst_row['legs_manhattan']}"
)

Both halves at once. It expands roughly half the vertices the admissible
heuristic does and about a twentieth of Dijkstra's — and it is wrong on close
to three fifths of the pairs. The size of the error is worth separating from
its frequency: the median excess is a fraction of a percent, the mean is a few
percent, and the worst is a quarter. So it is usually wrong by very little and
occasionally wrong by a lot, which is the harder failure to notice.

The worst pair is the shape of the failure rather than an outlier: the optimal
route is shorter *and* has fewer legs. A taxicab estimate pulls the search
toward vertices that are cheap to reach along the axes, closes the vertex the
good route ran through, and then cannot go back.

Note too that the excess stays well inside the bound section 2 set. The
worst-case ratio of 1.541 permits a returned cost of 54.1% over optimal; the
worst observed is a little under half of that. The bound is real but loose —
the estimate has to be wrong in the right places, not merely wrong.

### 4b. The same question, on five hand-picked pairs

The five long-haul pairs `experiments/search-cost` measures, reused verbatim so
this table lines up with that one's rather than describing five other queries.

They are here to show what the random sample is for. These are exactly the
pairs a person would choose to demonstrate an algorithm on — and most of them
are one direct flight, which no heuristic can get wrong.

In [ ]:
long_haul_rows, long_haul_unreachable = compare(
    tuple(pair) for pair in parameters["long_haul_pairs"]
)

for row in long_haul_rows:
    print(
        f"{row['origin']}-{row['destination']}  "
        f"optimal {row['optimal_km']:8,.1f} km in {row['legs_optimal']} leg(s)  "
        f"manhattan {row['manhattan_km']:8,.1f} km in {row['legs_manhattan']}  "
        f"excess {row['excess']:6.3%}  "
        f"expanded {row['expanded']}"
    )
if long_haul_unreachable:
    print(f"unreachable: {long_haul_unreachable}")

Three of the five are a single direct flight, where every algorithm expands one
vertex and returns the same answer. A hand-picked list of five would have
reported this heuristic as very nearly free and very nearly correct. The rate
in section 4 is the honest number, and the seed is recorded so it is
reproducible rather than merely repeatable.

### The trade, drawn

One point per pair: how much search it saved against how much optimality it
cost. The mass along the bottom is the queries it got right for free; the
spread above is what those cost.

In [ ]:
saved = [1 - row["expanded"]["manhattan"] / row["expanded"]["haversine"] for row in rows]
cost = [row["excess"] for row in rows]

figure, axes = plt.subplots(figsize=(7, 4.5))
axes.scatter(saved, cost, s=18, alpha=0.55, color="#2a78d6", edgecolor="none")
axes.axhline(0.0, color="#444444", linewidth=0.8)
axes.set_xlabel("expansions saved vs A* with the admissible heuristic")
axes.set_ylabel("cost above optimal")
axes.set_title(
    f"{len(rows)} random pairs: what the Manhattan heuristic buys, and what it costs"
)
axes.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes.spines[["top", "right"]].set_visible(False)
figure.tight_layout()
plt.show()

## The answer

Recorded next to the data that produced it.

In [ ]:
recorded = experiment.record(
    {
        "graph": {"airports": len(airports), "routes": len(routes)},
        "per_edge": per_edge,
        "consistency": consistency,
        "sample": sample,
        "long_haul": {
            "pairs": long_haul_rows,
            "unreachable": long_haul_unreachable,
        },
    },
    catalog=catalog,
)
print(recorded.name, experiment.results()["snapshot"]["id"])

## What this says

Manhattan distance on this graph overestimates the remaining cost on every edge
but a self-loop, by a third at the median. That makes A\* inconsistent, and an
inconsistent heuristic in a search that never reopens a closed vertex returns
suboptimal paths and reports them as optimal — here on close to three fifths of
random queries: by a fraction of a percent at the median, a few percent on
average, and a quarter at worst.

It is also genuinely faster, by about half the expansions of the admissible
heuristic. So this is not a broken formula; it is a trade, and one that a
route planner cannot take, because the thing it gives up is the only guarantee
the algorithm offers.

Set beside `heuristic-admissibility`, the pair makes a point neither makes
alone. Vincenty is a *more accurate* formula that breaks admissibility by tens
of kilometres. Manhattan is a *less accurate* one that breaks it by thousands.
Accuracy is not the axis: what matters is whether the estimate can exceed the
weights, and the weights here are haversine numbers by construction. Only
`haversine_heuristic()` is in that relationship with them, which is why it is
the only heuristic this project plans routes with.